In [ ]:
import pandas as pd
import numpy as np
import glob
import os

In [ ]:
# Solicitará permissão para acessar o seu Google Drive se for executado no Colab
from google.colab import drive
drive.mount('/content/drive')

# Definindo o caminho da nova pasta no Google Drive
caminho_pasta_base = '/content/drive/MyDrive/Fatec/TCC/dataset'
caminho_pasta_tratado = '/content/drive/MyDrive/Fatec/TCC/dataset tratado'

In [2]:
# Definindo os caminho da nova pasta no PC pessoal
caminho_pasta_base = './dataset'
caminho_pasta_tratado = './dataset tratado'

In [20]:
nome_dados_treinamento = 'cicids2017_treinamento.csv'
nome_dados_teste = 'cicids2017_teste.csv'

In [13]:
# Mapeia todos os arquivos que terminam com .csv dentro da pasta
arquivos_csv = glob.glob(os.path.join(caminho_pasta_base, "*.csv"))

print(f"Encontrados {len(arquivos_csv)} arquivos para importação.")

# Lê cada um dos 8 arquivos e os junta em um único DataFrame
lista_dfs = []
for arquivo in arquivos_csv:
    df_temp = pd.read_csv(arquivo)
    lista_dfs.append(df_temp)

# Concatena todos os DataFrames da lista empilhando as linhas
df = pd.concat(lista_dfs, ignore_index=True)

# Troca o label de ' Label' para 'Label' (removendo o espaço em branco)
df.rename(columns={' Label': 'Label'}, inplace=True) 

print("Tamanho total do dataset unificado:", df.shape)
display(df.head())

Encontrados 8 arquivos para importação.
Tamanho total do dataset unificado: (2830743, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [14]:
# Converte valores infinitos para NaN (nulos)
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Remove todas as linhas que contenham valores nulos
df.dropna(inplace=True)

print("Tamanho do dataset após remoção de nulos/infinitos:", df.shape)

Tamanho do dataset após remoção de nulos/infinitos: (2827876, 79)


In [23]:
from sklearn.preprocessing import MinMaxScaler

# Separando a variável alvo (Label) dos dados de tráfego
label = df['Label']
df = df.drop('Label', axis=1)

# Aplicação da escala Min-Max conforme exigido para convergência da Rede Neural
scaler = MinMaxScaler()
df = pd.DataFrame(
    scaler.fit_transform(df), 
    columns=df.columns,
    index=df.index
) 

# Readicionando a coluna 'Label' ao DataFrame tratado
df['Label'] = label.values

print("Dados numéricos normalizados com sucesso.")
display(df.head())

Dados numéricos normalizados com sucesso.


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0.837186,1.333333e-07,0.000005,0.000000,9.302326e-07,0.000000e+00,0.000242,0.002581,0.00101,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
1,0.840070,1.016667e-06,0.000000,0.000003,4.651163e-07,9.153974e-09,0.000242,0.002581,0.00101,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
2,0.840085,5.416666e-07,0.000000,0.000003,4.651163e-07,9.153974e-09,0.000242,0.002581,0.00101,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
3,0.705516,3.916666e-07,0.000000,0.000003,4.651163e-07,9.153974e-09,0.000242,0.002581,0.00101,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
4,0.837156,1.333333e-07,0.000005,0.000000,9.302326e-07,0.000000e+00,0.000242,0.002581,0.00101,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


In [ ]:
from sklearn.model_selection import train_test_split

df_treino, df_teste = train_test_split(df, test_size=0.3, random_state=42)

print(f"Dados de Treino: {df_treino.shape}")
print(f"Dados de Teste: {df_teste.shape}")

# O comando makedirs cria a pasta "dataset filtrado" caso ela ainda não exista no seu Drive
os.makedirs(caminho_pasta_tratado, exist_ok=True)

# 3. Caminho completo do arquivo CSV que será gerado
caminho_arquivo_treinamento = os.path.join(caminho_pasta_tratado, nome_dados_treinamento)
caminho_arquivo_teste = os.path.join(caminho_pasta_tratado, nome_dados_teste)

# 4. Exportando o DataFrame para o formato CSV
df_treino.to_csv(caminho_arquivo_treinamento, index=False)
print(f"Dataset de treinamento salvo com sucesso em: {caminho_arquivo_treinamento}")

df_teste.to_csv(caminho_arquivo_teste, index=False)
print(f"Dataset de teste salvo com sucesso em: {caminho_arquivo_teste}")

Dados de Treino: (1979513, 79)
Dados de Teste: (848363, 79)
Dataset de treinamento salvo com sucesso em: ./dataset tratado\cicids2017_treinamento.csv
Dataset de teste salvo com sucesso em: ./dataset tratado\cicids2017_teste.csv
